# A100 40GB limit test — Crowfeather-50M-v1

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Crownelius/crowfeather-50m-v1/blob/main/notebooks/a100_40gb_limit_test.ipynb)

Verifies that all three training phases fit on a 40GB A100 at the auto-adjusted batch sizes (B=2/1/2 with accum=4 for Phase 1/2/3). Uses **synthetic random token IDs** — no precache, no real tokenizer, no Drive I/O. Just tests the GPU envelope: model + optimizer + activations + logits.

## What this proves

- **Phase 1** (pretrain @ 4K, B=2, accum=4): expected ~12-15 GB peak. Easy fit.
- **Phase 2** (CPT @ 16K, B=1, accum=4): expected ~25-32 GB peak. The tightest test — 16K context with grad checkpointing.
- **Phase 3** (SFT @ 4K, B=2, accum=4): same envelope as Phase 1.

Each test runs 3 optimizer steps (with accumulation = 12 forward+backward passes total per phase) with synthetic data, measures peak VRAM, throughput, and confirms loss is finite. ~5 minutes total runtime.

## Required runtime

A100 40GB or 80GB. The test PASSES on either, but the diagnostic is most meaningful on 40GB (verifies the tighter envelope).


## 1. Setup

In [ ]:
# Mount Drive (so we can reuse main notebook's tokenizer/init if available, but not required)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/crowfeather_50m_v1'
print(f'Drive root (optional): {DRIVE_ROOT}')


In [ ]:
# Install deps. Conservative -U to bump older Colab cache.
import subprocess, sys

DEPS = [
    'transformers>=4.51',
    'tokenizers>=0.20',
    'accelerate',
    'liger-kernel',  # optional
]

for dep in DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', dep], check=False)

print('  done.')


In [ ]:
# Clone repo for muon.py
REPO_URL = 'https://github.com/Crownelius/crowfeather-50m-v1.git'
REPO_DIR = '/content/crowfeather-50m-v1'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

import sys
sys.path.insert(0, f'{REPO_DIR}/scripts')
print(f'Repo at: {REPO_DIR}')


## 2. GPU detection

Reports detected GPU and VRAM. The test runs on either 40GB or 80GB but the assertions target 40GB.


In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU detected. Runtime -> Change runtime type -> A100.'
gpu_name = torch.cuda.get_device_name(0)
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_80GB = gpu_total_gb > 60

VRAM_BUDGET_GB = 40 if not IS_80GB else 80
print(f'GPU: {gpu_name} ({gpu_total_gb:.1f} GB)')
print(f'Test target: {VRAM_BUDGET_GB} GB envelope')


## 3. Define the limit-test runner

Builds a fresh dense Qwen3 50M architecture from `Qwen3Config` (no tokenizer needed), constructs the Muon + AdamW hybrid optimizer, runs `n_optim_steps` steps with synthetic data, measures peak VRAM and throughput.


In [ ]:
import gc, time, random
import torch
from transformers import Qwen3Config, Qwen3ForCausalLM

VOCAB_SIZE = 32_768

CFG = dict(
    vocab_size=VOCAB_SIZE,
    hidden_size=448,
    num_hidden_layers=12,
    num_attention_heads=8,
    num_key_value_heads=4,
    head_dim=56,
    intermediate_size=1792,
    max_position_embeddings=16384,
    rope_theta=1_000_000.0,
    rms_norm_eps=1e-6,
    tie_word_embeddings=True,
    use_cache=False,
)


def build_fresh_model():
    cfg = Qwen3Config(**CFG)
    model = Qwen3ForCausalLM(cfg).to(torch.bfloat16).cuda()
    model.train()
    model.gradient_checkpointing_enable()
    return model


def run_limit_test(B, T, accum, n_optim_steps=3, peak_lr=3e-4, label='test', try_liger=True):
    """Build model + optimizer fresh, run n_optim_steps with synthetic data,
    return diagnostics. Cleans up GPU state on exit."""
    print(f'\n=== {label} : B={B} T={T} accum={accum} (eff_batch={B*accum}) ===')

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    model = build_fresh_model()
    n_params = sum(p.numel() for p in model.parameters())

    # Optionally apply Liger Kernel
    liger_status = 'OFF'
    if try_liger:
        try:
            from liger_kernel.transformers import apply_liger_kernel_to_qwen3
            apply_liger_kernel_to_qwen3(model)
            liger_status = 'ON'
        except ImportError:
            pass
        except Exception as e:
            liger_status = f'FAIL ({e})'

    # Build optimizers (uses repo's muon.py)
    from muon import build_optimizers
    muon_opt, adamw_opt = build_optimizers(model, peak_lr=peak_lr)
    optimizers = [o for o in [muon_opt, adamw_opt] if o is not None]

    # Synthetic data: random token IDs in valid range
    rng = torch.Generator(device='cuda')
    rng.manual_seed(20260504)

    t_start = time.time()
    losses = []
    n_tokens = 0
    for opt_step in range(n_optim_steps):
        for o in optimizers:
            o.zero_grad(set_to_none=True)
        for accum_i in range(accum):
            inp = torch.randint(0, VOCAB_SIZE, (B, T), device='cuda', dtype=torch.long, generator=rng)
            tgt = inp.clone()
            out = model(input_ids=inp, labels=tgt)
            loss = out.loss / accum
            loss.backward()
            losses.append(loss.item() * accum)
            n_tokens += B * T
            del out, loss, inp, tgt
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for o in optimizers:
            o.step()

    elapsed = time.time() - t_start
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    tps = n_tokens / elapsed
    avg_loss = sum(losses) / len(losses)
    finite = all(torch.isfinite(torch.tensor(l)).item() for l in losses)
    fits_40 = peak_gb < 38.5  # 40GB GPU has ~39.5 GB usable, leave 1 GB buffer

    print(f'  params:      {n_params/1e6:.2f}M')
    print(f'  Liger:       {liger_status}')
    print(f'  peak VRAM:   {peak_gb:.2f} GB / 40 GB target  ({100*peak_gb/40:.0f}%)')
    print(f'  throughput:  {tps:,.0f} tokens/sec')
    print(f'  avg loss:    {avg_loss:.3f}  finite: {finite}')
    print(f'  optimizer:   {n_optim_steps} steps in {elapsed:.1f}s ({elapsed/n_optim_steps:.2f}s/step)')
    verdict = 'PASS' if (fits_40 and finite) else 'FAIL'
    print(f'  verdict:     {verdict} ({"<40GB and finite" if verdict == "PASS" else "OOM or NaN"})')

    result = dict(
        label=label, B=B, T=T, accum=accum, eff_batch=B*accum,
        params_m=n_params/1e6, liger=liger_status,
        peak_vram_gb=peak_gb, tokens_per_sec=tps,
        avg_loss=avg_loss, finite=finite,
        elapsed_sec=elapsed, sec_per_step=elapsed/n_optim_steps,
        fits_40gb=fits_40, verdict=verdict,
    )

    # Cleanup
    del model, muon_opt, adamw_opt, optimizers
    gc.collect()
    torch.cuda.empty_cache()

    return result


## 4. Phase 1 limit test (pretrain @ 4K)

40GB-adjusted: B=2, accum=4, T=4096. Effective batch 8 = 32,768 tokens per optimizer step.

Expected: ~12-15 GB peak VRAM. Should be a comfortable fit.


In [ ]:
r1 = run_limit_test(B=2, T=4096, accum=4, n_optim_steps=3, peak_lr=3e-4, label='phase1_40gb_pretrain')


## 5. Phase 2 limit test (CPT @ 16K)

40GB-adjusted: B=1, accum=4, T=16384. Effective batch 4 = 65,536 tokens per optimizer step.

**This is the tightest test.** 16K context + grad checkpointing. Expected: ~25-32 GB peak VRAM. If this fails on 40GB, the long-context phase is over budget.


In [ ]:
r2 = run_limit_test(B=1, T=16384, accum=4, n_optim_steps=3, peak_lr=6e-5, label='phase2_40gb_cpt_16k')


## 6. Phase 3 limit test (SFT @ 4K)

40GB-adjusted: B=2, accum=4, T=4096. Same memory envelope as Phase 1. Should pass identically.


In [ ]:
r3 = run_limit_test(B=2, T=4096, accum=4, n_optim_steps=3, peak_lr=4e-5, label='phase3_40gb_sft')


## 7. Summary report

Pretty-print all three test results. PASS = fits in 40 GB AND loss is finite.


In [ ]:
results = [r1, r2, r3]

print(f'\n{"="*78}')
print(f'A100 40GB LIMIT TEST SUMMARY')
print(f'{"="*78}\n')
print(f'{"phase":30} {"B":>3} {"T":>6} {"accum":>5} {"eff":>4} {"VRAM":>8} {"tps":>10} {"verdict":>8}')
print(f'{"-"*78}')
for r in results:
    print(f'{r["label"]:30} {r["B"]:>3} {r["T"]:>6} {r["accum"]:>5} {r["eff_batch"]:>4} '
          f'{r["peak_vram_gb"]:>6.1f}GB {r["tokens_per_sec"]:>9,.0f}/s {r["verdict"]:>8}')
print(f'{"-"*78}')

all_pass = all(r['verdict'] == 'PASS' for r in results)
print(f'\nOVERALL: {"ALL PHASES PASS" if all_pass else "AT LEAST ONE PHASE FAILED"}')
if all_pass:
    print('  -> 40GB A100 is sufficient for full Crowfeather-50M-v1 training.')
    print('  -> Open the main training notebook and run all cells.')
else:
    failed = [r['label'] for r in results if r['verdict'] != 'PASS']
    print(f'  -> Failed phases: {failed}')
    print(f'  -> Either upgrade to 80GB A100, or reduce batch size further (manual edit in train_dense.py).')

# Wall-time projection from throughput
print(f'\n{"="*78}')
print(f'WALL-TIME PROJECTION (extrapolated from 3-step throughput)')
print(f'{"="*78}\n')
phase_steps = {'phase1_40gb_pretrain': 40_000, 'phase2_40gb_cpt_16k': 2_500, 'phase3_40gb_sft': 2_500}
total_h = 0.0
for r in results:
    steps = phase_steps[r['label']]
    proj_sec = steps * r['sec_per_step']
    proj_h = proj_sec / 3600
    total_h += proj_h
    print(f'  {r["label"]:30} {steps:>6} steps @ {r["sec_per_step"]:.2f}s/step = {proj_h:.1f}h')
print(f'  {"-"*70}')
print(f'  {"TOTAL projected training":30} {total_h:.1f}h (+ ~1h precache and BPE)')
print(f'\nAt Colab Pro PAYG (~$1.50/hr A100), expected cost: ${total_h * 1.5:.0f}-${(total_h+2) * 1.5:.0f}.')


## 8. (Optional) Stress test — Phase 2 at higher batch

If Phase 2 passed comfortably (<25 GB peak), try B=2 to see if it still fits. Useful if you have an 80GB A100 and want to halve Phase 2 wall time.


In [ ]:
# Only run this if Phase 2 above passed AND you want to see if a higher batch fits
if r2['verdict'] == 'PASS' and r2['peak_vram_gb'] < 25:
    print('Phase 2 had comfortable headroom; trying B=2 T=16384 stress test...')
    r2_stress = run_limit_test(B=2, T=16384, accum=2, n_optim_steps=3, peak_lr=6e-5, label='phase2_stress_B2')
    if r2_stress['verdict'] == 'PASS':
        print(f'\n  STRESS PASS: B=2 T=16K fits at {r2_stress["peak_vram_gb"]:.1f} GB.')
        print(f'  -> Edit train_dense.py phase2 args to use B=2 accum=2 for ~2x faster Phase 2.')
    else:
        print(f'\n  STRESS FAIL at B=2; keep B=1 accum=4 for Phase 2.')
else:
    print('Skipped stress test (Phase 2 not enough headroom or 80GB user).')
